# Figure 4: hierarchical concept analysis

Score and visualize single-level and PCT-refined concepts in human pancreas and lung datasets. Run the setup, pancreas analysis, and lung analysis in order.

## Prerequisites

- Install scConcept and prepare processed datasets using Tutorials 0–3.
- Generate the required concepts first, or use the deposited concept files in `Tutorial2/Concept`.
- Configure the input paths below. Processed `.mat` files use `Datasets/`; additional clinical, annotation, or plotting inputs are required by the corresponding analysis.

This tutorial starts from saved concepts. Run cells from top to bottom; preprocessing, topic training, and concept generation are not repeated. The shared input helpers are in `Tutorial2/tutorial_utils.py`.


In [ ]:
from pathlib import Path
import json
import hashlib
import gc
import platform
import importlib.metadata
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.io import loadmat
import scanpy as sc
import anndata as ad
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
from sklearn.cluster import KMeans
from IPython.display import display

# =========================================================
# Locate the installed scConcept repository and tutorial inputs
# =========================================================
import os
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scConcept.py").is_file() and (p / "Tutorial2").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the scConcept repository or one of its subdirectories.")
TUTORIAL_DIR = PROJECT_ROOT / "Tutorial2"
for directory in [PROJECT_ROOT, TUTORIAL_DIR]:
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from tutorial_utils import (ScConcept, SingleCellDataset, configure_inputs,
    input_path, dataset_path, load_mat_test, load_api_dataset, load_saved_concepts,
    score_saved_concepts, matlab_string, runtime_paths,
    compute_coherence_from_gene_lists, TD_eva_topics, read_pathways, interpretability)
PACKAGE_DATA_DIR = configure_inputs()
REPO_DIR = PROJECT_ROOT
CONCEPT_DIR = TUTORIAL_DIR / "Concept"

# Dataset paths can be overridden in Inputs/data_paths.local.json.
OUTPUT_DIR = TUTORIAL_DIR / "outputs/figure4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



PANCREAS_MAT = dataset_path("Pancreas_10k")
LUNG_MAT = dataset_path("Lung_30k", "Lung_10k")
PANCREAS_ANNOTATIONS = input_path("hierarchy.pancreas_annotations", "Pancreas.h5ad")
LUNG_ANNOTATIONS = input_path("hierarchy.lung_annotations", "Lung.h5ad")
GMT_PATH = input_path("gmt.Hs", "msigdb.v2024.1.Hs.symbols.gmt")
SCORING_TOP_K = 100
METRIC_TOP_K = 10
RUN_UMAP = True
RUN_TC_TD = True
RUN_PCT_DIAGNOSTICS = False
PANCREAS_PCT = dict(tau_frac=0.5, min_cells=30, min_leaf=20, min_impurity_reduction=0.1, random_state=0)
LUNG_PCT = dict(tau_frac=0.6, min_cells=80, min_leaf=50, min_impurity_reduction=0.15, random_state=0)
PANCREAS_ZOOM_BOUNDS = (4, 9.0, 15.5, 20.5)
PANCREAS_ZOOM_MIN_CELLS = 50


## Shared functions

Score concepts using the mean gene-wise z-score of their first 100 listed genes. Assign each cell to its highest-scoring parent, then to its highest-scoring child within that parent.

Compute TC and TD from the first 10 genes per concept. TD counts genes occurring exactly once; `TD_unique_fraction` reports the distinct-symbol fraction. Report NMI and AMI separately. Optional PCT diagnostics evaluate expression heterogeneity using the configured thresholds.

The helpers below retain the reported TC/TD definitions. They are evaluated from each current concept set, rather than reusing cached annotations from `ScConcept.evaluation()`.


In [ ]:



def load_expression_and_labels(mat_path, annotation_path):
    for path in [mat_path, annotation_path]:
        if not path.is_file():
            raise FileNotFoundError(path)
    expression, genes, coarse_labels = load_mat_test(mat_path, dtype=np.float32)
    if coarse_labels is None:
        raise ValueError("The hierarchy dataset requires label_test.")
    reference = sc.read_h5ad(annotation_path, backed="r")
    try:
        obs = reference.obs.copy()
    finally:
        reference.file.close()
    if expression.shape != (len(coarse_labels), len(genes)) or len(obs) != len(coarse_labels):
        raise ValueError("Expression, coarse labels, and fine annotation rows are not aligned.")
    if not obs.index.is_unique or len(set(genes)) != len(genes):
        raise ValueError("Cell identifiers and gene symbols must be unique.")
    if "Curated_annotation" not in obs:
        raise KeyError("Expected Curated_annotation in the annotation file.")
    if obs["Curated_annotation"].isna().any():
        raise ValueError("Missing fine-grained annotations.")

    fine_labels = obs["Curated_annotation"].astype(str).to_numpy()
    return expression, genes, coarse_labels, fine_labels, obs.index.astype(str).to_numpy()


def load_concept_hierarchy(parent_path, child_path):
    parents = load_saved_concepts(parent_path)
    nested = json.loads(child_path.read_text())
    # ScConcept.hconcept() saves the refined entries inside a result dictionary.
    if isinstance(nested, dict):
        nested = nested.get("refined_results")
    if not isinstance(nested, list):
        raise ValueError("Expected a refined-results list or a ScConcept hierarchical output.")
    if len(parents) != len(nested):
        raise ValueError("Parent and hierarchy entry counts differ.")
    for i, (parent, entry) in enumerate(zip(parents, nested)):
        if entry.get("concept_name", parent["name"]) != parent["name"]:
            raise ValueError(f"Parent name/order mismatch at entry {i}.")
        children = entry.get("sub_concepts", [])
        if not children:
            raise ValueError(f"No child concept for parent {parent['name']!r}.")
        for child in children:
            if not child.get("genes") or not child.get("name"):
                raise ValueError("Child concepts require a name and nonempty gene list.")
            if not set(child["genes"]).issubset(parent["genes"]):
                raise ValueError(f"Child genes lie outside parent {parent['name']!r}.")
    return parents, nested


def score_concepts(concepts, expression, genes, top_k=100):
    """Use ScConcept annotation while retaining sparse atlas inputs."""
    return score_saved_concepts(concepts, expression, genes, top_k)


In [ ]:
def flatten_subconcepts_minimal(refined_results):
    """
    Convert GPT hierarchical output into:
    1. A flat list of subconcept gene lists.
    2. A parent_ids array that maps each subconcept back to its parent old concept.

    refined_results: list of dicts
      Each element corresponds to one old concept:
      {
         "parent_concept": {...},
         "sub_concepts": [
             {"name": ..., "genes": [...], "description": ...},
             ...
         ]
      }

    Returns:
      flat_genes: list of lists  (each is a subconcept's genes)
      parent_ids_for_new: np.ndarray, shape (n_new,)
    """
    flat_concepts = []
    parent_ids_for_new = []
    for (parent_idx, result) in enumerate(refined_results):
        subcs = result['sub_concepts']
        for sc in subcs:
            flat_concepts.append({'name': sc['name'], 'genes': sc['genes']})
            parent_ids_for_new.append(parent_idx)
    return (flat_concepts, np.array(parent_ids_for_new))


In [ ]:
def hierarchical_assign(score_old, score_new, parent_ids_for_new, old_concept_names=None, new_concept_names=None, old_threshold=None, new_threshold=None):
    """Assign each cell to its best parent and the highest-scoring child within that parent."""
    (n_cells, n_old) = score_old.shape
    (_, n_new) = score_new.shape
    parent_ids_for_new = np.asarray(parent_ids_for_new)
    assert parent_ids_for_new.shape[0] == n_new, 'Parent map length must equal the number of child concepts.'
    old_best_idx = np.argmax(score_old, axis=1)
    old_best_score = score_old[np.arange(n_cells), old_best_idx]
    new_best_idx = np.full(n_cells, -1, dtype=int)
    new_best_score = np.full(n_cells, np.nan)
    for i in range(n_cells):
        parent = old_best_idx[i]
        if old_threshold is not None and old_best_score[i] < old_threshold:
            new_best_idx[i] = -1
            new_best_score[i] = np.nan
            continue
        child_mask = parent_ids_for_new == parent
        child_indices = np.where(child_mask)[0]
        if child_indices.size == 0:
            new_best_idx[i] = -1
            new_best_score[i] = np.nan
            continue
        scores_for_children = score_new[i, child_indices]
        local_best_pos = np.argmax(scores_for_children)
        child_global_idx = child_indices[local_best_pos]
        child_score = scores_for_children[local_best_pos]
        if new_threshold is not None and child_score < new_threshold:
            new_best_idx[i] = -1
            new_best_score[i] = np.nan
        else:
            new_best_idx[i] = child_global_idx
            new_best_score[i] = child_score
    df = pd.DataFrame({'old_concept_idx': old_best_idx, 'score_old_max': old_best_score, 'new_concept_idx': new_best_idx, 'score_new_max_in_parent': new_best_score})
    if old_concept_names is not None:
        df['old_concept_name'] = [old_concept_names[idx] for idx in old_best_idx]
    if new_concept_names is not None:
        name_col = []
        for idx in new_best_idx:
            if idx == -1:
                name_col.append(None)
            else:
                name_col.append(new_concept_names[idx])
        df['new_concept_name'] = name_col
    return df


In [ ]:
def pct_should_split_concept_expr_matrix(X, vocab, concepts, score_old, concept_idx, tau_frac=0.5, min_cells=30, min_leaf=20, min_impurity_reduction=0.1, random_state=0):
    """
    Expression-based PCT-style split test using the input expression matrix.
    NO labels, NO AnnData.
    """
    info = {'concept_idx': int(concept_idx)}
    gene_to_idx = {g: i for (i, g) in enumerate(vocab)}
    s_all = score_old[:, concept_idx]
    tau = tau_frac * float(np.max(s_all))
    cell_idx = np.where(s_all > tau)[0]
    n_cells = len(cell_idx)
    info.update({'tau': tau, 'n_cells': n_cells})
    if n_cells < min_cells:
        info['reason'] = 'too_few_cells'
        return (False, info)
    concept_genes = concepts[concept_idx]['genes']
    gene_indices = [gene_to_idx[g] for g in concept_genes if g in gene_to_idx]
    info['n_genes_in_data'] = len(gene_indices)
    if len(gene_indices) < 10:
        info['reason'] = 'too_few_genes'
        return (False, info)
    if hasattr(X, 'detach'):
        X_sub = X[cell_idx][:, gene_indices].detach().cpu().numpy()
    else:
        X_sub = X[np.ix_(cell_idx, gene_indices)]
    X_sub = X_sub.toarray() if sparse.issparse(X_sub) else X_sub
    X_sub = X_sub.astype(float)
    mu = X_sub.mean(axis=0, keepdims=True)
    parent_sse = float(np.sum((X_sub - mu) ** 2))
    info['parent_sse'] = parent_sse
    if parent_sse < 1e-08:
        info['reason'] = 'near_zero_variance'
        return (False, info)
    km = KMeans(n_clusters=2, random_state=random_state, n_init=10)
    labels = km.fit_predict(X_sub)
    idx0 = labels == 0
    idx1 = labels == 1
    (n0, n1) = (idx0.sum(), idx1.sum())
    info['cluster_sizes'] = (int(n0), int(n1))
    if min(n0, n1) < min_leaf:
        info['reason'] = 'small_child_cluster'
        return (False, info)
    mu0 = X_sub[idx0].mean(axis=0, keepdims=True)
    mu1 = X_sub[idx1].mean(axis=0, keepdims=True)
    sse0 = float(np.sum((X_sub[idx0] - mu0) ** 2))
    sse1 = float(np.sum((X_sub[idx1] - mu1) ** 2))
    child_sse = sse0 + sse1
    impurity_reduction = (parent_sse - child_sse) / parent_sse
    info['impurity_reduction'] = float(impurity_reduction)
    if impurity_reduction < min_impurity_reduction:
        info['reason'] = 'low_impurity_reduction'
        return (False, info)
    info['reason'] = 'PASS'
    return (True, info)


In [ ]:




def evaluate_levels(coarse, fine, scores, child_scores, assignments, parents, children, genes, pathways):
    rows = []
    for level, truth, prediction, definitions in [
        ("single_level", coarse, scores.argmax(axis=1), parents),
        ("hierarchical", fine, assignments["new_concept_idx"].to_numpy(), children)]:
        row = dict(level=level, n_cells=len(truth), n_concepts=len(definitions),
                   ARI=adjusted_rand_score(truth, prediction),
                   NMI=normalized_mutual_info_score(truth, prediction),
                   AMI=adjusted_mutual_info_score(truth, prediction))
        if pathways is not None:
            row.update(interpretability(definitions, genes, pathways, METRIC_TOP_K))
        rows.append(row)
    return pd.DataFrame(rows)


def pct_diagnostics(expression, genes, parents, scores, settings):
    rows = []
    for i, parent in enumerate(parents):
        should_split, info = pct_should_split_concept_expr_matrix(
            expression, genes, parents, scores, i, **settings)
        rows.append(dict(concept=parent["name"], should_split=bool(should_split), **info))
    return pd.DataFrame(rows)


def expression_umap(expression, genes, cell_ids, coarse, fine, scores, names, assignments):
    dense = expression.toarray() if sparse.issparse(expression) else np.asarray(expression)
    data = ad.AnnData(dense)
    data.obs_names = cell_ids
    data.var_names = genes
    data.obs["cell_type"] = pd.Categorical(coarse)
    data.obs["dominant_concept"] = pd.Categorical(np.asarray(names)[scores.argmax(axis=1)])
    data.obs["type_label_level"] = pd.Categorical(fine)
    data.obs["concept_label_level"] = pd.Categorical(assignments["new_concept_name"].to_numpy())
    sc.tl.pca(data)
    sc.pp.neighbors(data)
    sc.tl.umap(data)
    return data


def plot_annotation_pair(data, keys, ncols, path):
    fig = sc.pl.umap(data, color=keys, ncols=ncols, frameon=False,
                     wspace=0.35, legend_loc="right margin", return_fig=True, show=False)
    for ax in fig.axes:
        legend = ax.legend_
        if legend is not None:
            for text in legend.get_texts():
                text.set_fontsize(14)
            handles = getattr(legend, "legend_handles", getattr(legend, "legendHandles", []))
            for handle in handles:
                if hasattr(handle, "set_sizes"):
                    handle.set_sizes([60])
    fig.savefig(path, bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig)


def save_dataset_outputs(out_dir, scores, child_scores, assignments, metrics, cell_ids, coarse, fine, names, child_names, parent_ids, matches):
    out_dir.mkdir(parents=True, exist_ok=True)
    assignments = assignments.copy()
    assignments.insert(0, "cell_id", cell_ids)
    assignments["coarse_ground_truth"] = coarse
    assignments["fine_ground_truth"] = fine
    assignments.to_csv(out_dir / "cell_assignments.csv", index=False)
    metrics.to_csv(out_dir / "metrics.csv", index=False)
    np.savez_compressed(out_dir / "concept_scores.npz", single_level=scores, hierarchical=child_scores,
                        parent_names=np.array(names), child_names=np.array(child_names), parent_ids=parent_ids)
    (out_dir / "gene_matches.json").write_text(json.dumps(matches, indent=2))

PATHWAYS = read_pathways(GMT_PATH) if RUN_TC_TD else None


## 1. Pancreas

Load expression, coarse labels, fine labels (`Curated_annotation`), and saved concepts. Expression and annotation rows must be in matching cell order.


In [ ]:
pancreas_out = OUTPUT_DIR / "Pancreas"
pancreas_out.mkdir(exist_ok=True)
pancreas_X, pancreas_genes, pancreas_coarse, pancreas_fine, pancreas_ids = load_expression_and_labels(PANCREAS_MAT, PANCREAS_ANNOTATIONS)
pancreas_parents, pancreas_nested = load_concept_hierarchy(
    CONCEPT_DIR / "GPT5_Pancreas_10k_level1_concepts.json",
    CONCEPT_DIR / "GPT5_Pancreas_10k_level2_concepts.json")
pancreas_children, pancreas_parent_ids = flatten_subconcepts_minimal(pancreas_nested)
print("Pancreas expression:", pancreas_X.shape)
print("Annotation levels:", len(set(pancreas_coarse)), len(set(pancreas_fine)))
print("Concept levels:", len(pancreas_parents), len(pancreas_children))


In [ ]:
pancreas_scores, pancreas_names, pancreas_matches_l1 = score_concepts(pancreas_parents, pancreas_X, pancreas_genes, SCORING_TOP_K)
pancreas_child_scores, pancreas_child_names, pancreas_matches_l2 = score_concepts(pancreas_children, pancreas_X, pancreas_genes, SCORING_TOP_K)
pancreas_assignments = hierarchical_assign(pancreas_scores, pancreas_child_scores, pancreas_parent_ids,
    old_concept_names=pancreas_names, new_concept_names=pancreas_child_names)
assert (pancreas_assignments["new_concept_idx"] >= 0).all()
pancreas_metrics = evaluate_levels(pancreas_coarse, pancreas_fine, pancreas_scores, pancreas_child_scores,
    pancreas_assignments, pancreas_parents, pancreas_children, pancreas_genes, PATHWAYS)
display(pancreas_metrics)
save_dataset_outputs(pancreas_out, pancreas_scores, pancreas_child_scores, pancreas_assignments, pancreas_metrics,
    pancreas_ids, pancreas_coarse, pancreas_fine, pancreas_names, pancreas_child_names, pancreas_parent_ids,
    dict(single_level=pancreas_matches_l1, hierarchical=pancreas_matches_l2))
if RUN_PCT_DIAGNOSTICS:
    pct_diagnostics(pancreas_X, pancreas_genes, pancreas_parents, pancreas_scores, PANCREAS_PCT).to_csv(
        pancreas_out / "pct_diagnostics.csv", index=False)


### Pancreas annotation maps

Compute one expression UMAP and use it to display ground-truth labels and concept assignments at both levels.


In [ ]:
if RUN_UMAP:
    pancreas_adata = expression_umap(pancreas_X, pancreas_genes, pancreas_ids, pancreas_coarse, pancreas_fine,
                                    pancreas_scores, pancreas_names, pancreas_assignments)
    plot_annotation_pair(pancreas_adata, ["cell_type", "dominant_concept"], 2, pancreas_out / "Pancreas_level1.pdf")
    plot_annotation_pair(pancreas_adata, ["type_label_level", "concept_label_level"], 2, pancreas_out / "Pancreas_level2.pdf")
    np.savez_compressed(pancreas_out / "expression_umap.npz", coordinates=pancreas_adata.obsm["X_umap"], cell_ids=pancreas_ids)


### Pancreatic endocrine zoom-in

Crop the existing UMAP using `PANCREAS_ZOOM_BOUNDS`. Retain cells whose concept and fine-label categories each contain at least `PANCREAS_ZOOM_MIN_CELLS` cells within the crop. Plot sub-concepts and fine labels using the same coordinates.


In [ ]:
if RUN_UMAP:
    x_min, x_max, y_min, y_max = PANCREAS_ZOOM_BOUNDS
    coords = pancreas_adata.obsm["X_umap"]
    crop = ((coords[:, 0] >= x_min) & (coords[:, 0] <= x_max) &
            (coords[:, 1] >= y_min-1) & (coords[:, 1] <= y_max))
    pancreas_zoom = pancreas_adata[crop].copy()
    concept_counts = pancreas_zoom.obs["concept_label_level"].value_counts()
    type_counts = pancreas_zoom.obs["type_label_level"].value_counts()
    valid_concepts = concept_counts[concept_counts >= PANCREAS_ZOOM_MIN_CELLS].index
    valid_types = type_counts[type_counts >= PANCREAS_ZOOM_MIN_CELLS].index
    keep = (pancreas_zoom.obs["concept_label_level"].isin(valid_concepts) &
            pancreas_zoom.obs["type_label_level"].isin(valid_types))
    pancreas_zoom = pancreas_zoom[keep].copy()
    if not pancreas_zoom.n_obs:
        raise ValueError("The endocrine crop is empty; check the embedding and coordinate bounds.")
    for column in ["concept_label_level", "type_label_level"]:
        pancreas_zoom.obs[column] = pancreas_zoom.obs[column].cat.remove_unused_categories()
    assert np.array_equal(pancreas_zoom.obsm["X_umap"], coords[crop][keep.to_numpy()])
    print("Endocrine zoom cells:", pancreas_zoom.n_obs)
    display(pd.crosstab(pancreas_zoom.obs["concept_label_level"], pancreas_zoom.obs["type_label_level"]))
    zoom_palettes = {
        "concept_label_level": {
            "Alpha-cell glucagon signaling": "#4C72B0",
            "Beta-cell insulin secretion": "#DD8452",
            "Delta/PP neuroendocrine peptides": "#55A868",
        },
        "type_label_level": {
            "Alpha cells": "#4C72B0", "Beta cells": "#DD8452",
            "Delta cells": "#55A868", "Gamma cells": "#C44E52",
        },
    }
    for column, filename in [("concept_label_level", "Pancreas_concept_label.pdf"),
                             ("type_label_level", "Pancreas_type_label.pdf")]:

        categories = pancreas_zoom.obs[column].cat.categories
        base = plt.get_cmap("tab20")
        palette = {name: zoom_palettes[column].get(name, base(i % 20))
                   for i, name in enumerate(categories)}
        fig = sc.pl.umap(pancreas_zoom, color=column, palette=palette, frameon=False,
                         legend_loc="right margin", return_fig=True, show=False)
        fig.savefig(pancreas_out / filename, bbox_inches="tight", format="pdf")
        plt.show()
        plt.close(fig)
    pancreas_zoom.obs.to_csv(pancreas_out / "endocrine_zoom_annotations.csv")
    np.savez_compressed(pancreas_out / "endocrine_zoom_coordinates.npz",
                        coordinates=pancreas_zoom.obsm["X_umap"], cell_ids=pancreas_zoom.obs_names.to_numpy(dtype=str))


In [ ]:

del pancreas_X
if RUN_UMAP:
    del pancreas_adata, pancreas_zoom
_ = gc.collect()


## 2. Lung

Load the lung expression, matching annotations, and saved single-level and hierarchical concepts. Calculate metrics and visualize both annotation levels.


In [ ]:
lung_out = OUTPUT_DIR / "Lung"
lung_out.mkdir(exist_ok=True)
lung_X, lung_genes, lung_coarse, lung_fine, lung_ids = load_expression_and_labels(LUNG_MAT, LUNG_ANNOTATIONS)
lung_parents, lung_nested = load_concept_hierarchy(
    CONCEPT_DIR / "GPT5_Lung_30k_level1_concepts.json",
    CONCEPT_DIR / "GPT5_Lung_30k_level2_concepts.json")
lung_children, lung_parent_ids = flatten_subconcepts_minimal(lung_nested)
print("Lung expression:", lung_X.shape)
print("Annotation levels:", len(set(lung_coarse)), len(set(lung_fine)))
print("Concept levels:", len(lung_parents), len(lung_children))


In [ ]:
lung_scores, lung_names, lung_matches_l1 = score_concepts(lung_parents, lung_X, lung_genes, SCORING_TOP_K)
lung_child_scores, lung_child_names, lung_matches_l2 = score_concepts(lung_children, lung_X, lung_genes, SCORING_TOP_K)
lung_assignments = hierarchical_assign(lung_scores, lung_child_scores, lung_parent_ids,
    old_concept_names=lung_names, new_concept_names=lung_child_names)
assert (lung_assignments["new_concept_idx"] >= 0).all()
lung_metrics = evaluate_levels(lung_coarse, lung_fine, lung_scores, lung_child_scores,
    lung_assignments, lung_parents, lung_children, lung_genes, PATHWAYS)
display(lung_metrics)
save_dataset_outputs(lung_out, lung_scores, lung_child_scores, lung_assignments, lung_metrics,
    lung_ids, lung_coarse, lung_fine, lung_names, lung_child_names, lung_parent_ids,
    dict(single_level=lung_matches_l1, hierarchical=lung_matches_l2))
if RUN_PCT_DIAGNOSTICS:
    pct_diagnostics(lung_X, lung_genes, lung_parents, lung_scores, LUNG_PCT).to_csv(
        lung_out / "pct_diagnostics.csv", index=False)

In [ ]:
if RUN_UMAP:
    lung_adata = expression_umap(lung_X, lung_genes, lung_ids, lung_coarse, lung_fine,
                               lung_scores, lung_names, lung_assignments)
    plot_annotation_pair(lung_adata, ["cell_type", "dominant_concept"], 1, lung_out / "Lung_level1.pdf")
    plot_annotation_pair(lung_adata, ["type_label_level", "concept_label_level"], 1, lung_out / "Lung_level2.pdf")
    np.savez_compressed(lung_out / "expression_umap.npz", coordinates=lung_adata.obsm["X_umap"], cell_ids=lung_ids)


## Output guide

Paths below are relative to `outputs/figure4`. Outputs are saved separately for Pancreas and Lung.

| Analysis | Main output |
|---|---|
| Pancreas coarse-level annotations | `Pancreas/Pancreas_level1.pdf` |
| Pancreas fine-level annotations | `Pancreas/Pancreas_level2.pdf` |
| Pancreatic endocrine sub-concept zoom | `Pancreas/Pancreas_concept_label.pdf` |
| Pancreatic endocrine cell-type zoom | `Pancreas/Pancreas_type_label.pdf` |
| Lung coarse-level annotations | `Lung/Lung_level1.pdf` |
| Lung fine-level annotations | `Lung/Lung_level2.pdf` |
| Single-level and hierarchical ARI, NMI, AMI, TC, and TD | `<dataset>/metrics.csv` |
| Cell assignments and concept scores | `<dataset>/cell_assignments.csv`, `<dataset>/concept_scores.npz` |
| Expression UMAP coordinates | `<dataset>/expression_umap.npz` |
| Optional PCT diagnostics | `<dataset>/pct_diagnostics.csv` |

`<dataset>` is `Pancreas` or `Lung`. The final cell saves the run configuration and package versions.


## Save run settings

Save metrics, assignments, scores, and figures separately under `outputs/figure4/Pancreas` and `outputs/figure4/Lung`, together with input paths, parameters, and package versions.


In [ ]:
versions = {}
for package in ["numpy", "pandas", "scipy", "scikit-learn", "scanpy", "anndata", "umap-learn", "matplotlib"]:
    versions[package] = importlib.metadata.version(package)
concept_files = [CONCEPT_DIR / name for name in [
    "GPT5_Pancreas_10k_level1_concepts.json", "GPT5_Pancreas_10k_level2_concepts.json",
    "GPT5_Lung_30k_level1_concepts.json", "GPT5_Lung_30k_level2_concepts.json"]]
settings = dict(python=platform.python_version(), packages=versions,
    input_paths=dict(pancreas_expression=str(PANCREAS_MAT), pancreas_annotations=str(PANCREAS_ANNOTATIONS),
                     lung_expression=str(LUNG_MAT), lung_annotations=str(LUNG_ANNOTATIONS), pathways=str(GMT_PATH)),
    concept_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in concept_files},
    scoring_top_k=SCORING_TOP_K, metric_top_k=METRIC_TOP_K,
    pct=dict(pancreas=PANCREAS_PCT, lung=LUNG_PCT),
    pancreas_zoom_bounds=PANCREAS_ZOOM_BOUNDS, pancreas_zoom_min_cells=PANCREAS_ZOOM_MIN_CELLS,
    enabled=dict(umap=RUN_UMAP, tc_td=RUN_TC_TD, pct_diagnostics=RUN_PCT_DIAGNOSTICS))
settings["package_paths"] = runtime_paths()
(OUTPUT_DIR / "run_configuration.json").write_text(json.dumps(settings, indent=2))
print("Outputs:", OUTPUT_DIR)
